In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm


from paddy_10_data_loader import load_train_val_data
from shufflenet_v2 import ShuffleNetV2

from kd_utils import student_training_loop, evaluate
from helper_utils import count_params


In [22]:
train_loader, val_loader = load_train_val_data(batch_size=32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## ShuffleNetV2 as a student model with knowledge distillation

In [23]:
densenet_201_teacher = timm.create_model("densenet201", pretrained=False, num_classes=10)
densenet_201_teacher.load_state_dict(torch.load("densenet_201_teacher.pth"))

<All keys matched successfully>

In [24]:
shufflenet_v2_student = ShuffleNetV2(n_class=10, model_size="0.90x")

model size is  0.90x


In [25]:
count_params(shufflenet_v2_student)

Total Parameters:         1,191,708
Trainable Parameters:     1,176,232
Non-trainable Parameters: 15,476


In [26]:
epochs = 15

In [ ]:
shufflenet_v2_student_loss = nn.CrossEntropyLoss()
shufflenet_v2_student_optimizer = optim.Adam(shufflenet_v2_student.parameters(), lr=0.001)

In [ ]:
trained_shufflenet_v2_student, shufflenet_v2_student_history = student_training_loop(
   teacher=densenet_201_teacher,
   student=shufflenet_v2_student,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=shufflenet_v2_student_optimizer,
    temperature=4,
    alpha=0.90,
    num_epochs=15,
    device=device,
    scheduler=None,
    save_path= "shufflenet_v2_student.pth",
)

Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 31.3298%, Train Hard Loss: 1.9754, Train Soft Loss: 1.0039, Train Distill Loss: 3.3841 | Val Hard Loss: 1.9471, Val Soft Loss: 0.8496, Val Distill Loss: 7.7700, Val Acc: 34.2926%
Epoch 2/15 - Train Acc: 48.5950%, Train Hard Loss: 1.5296, Train Soft Loss: 0.7927, Train Distill Loss: 2.6449 | Val Hard Loss: 1.9685, Val Soft Loss: 0.6747, Val Distill Loss: 6.3821, Val Acc: 47.6619%
Epoch 3/15 - Train Acc: 59.9699%, Train Hard Loss: 1.2272, Train Soft Loss: 0.6406, Train Distill Loss: 2.1294 | Val Hard Loss: 1.3582, Val Soft Loss: 0.5165, Val Distill Loss: 4.8113, Val Acc: 59.9520%
Epoch 4/15 - Train Acc: 69.5116%, Train Hard Loss: 0.9826, Train Soft Loss: 0.5211, Train Distill Loss: 1.7182 | Val Hard Loss: 1.4183, Val Soft Loss: 0.5111, Val Distill Loss: 4.7980, Val Acc: 61.0911%
Epoch 5/15 - Train Acc: 75.2066%, Train Hard Loss: 0.8028, Train Soft Loss: 0.4196, Train Distill Loss: 1.3938 | Val Hard Loss: 0.9697, Val Soft Loss: 0.3692, Val Distill Loss: 3.4388, Val